In [ ]:
from src.tokenizers.BPE_v2 import BytePairEncoding

from datasets import load_dataset
from src.utils import paths

In [ ]:
tokenizer = BytePairEncoding.from_file(paths.EXPERIMENTS_DIR / "classla_finewebedu_2gb_tokenizer_Bytes")

In [ ]:


hf_dataset = load_dataset(
    "wikimedia/wikipedia", 
    "20231101.hr", 
    cache_dir=paths.DATA_DIR / "wikimedia/wikipedia/20231101/hr",
)


In [ ]:
words = []
tokens = []

i = 1000
for sample in hf_dataset["train"]:
    i -= 1
    if i == 0:
        break
    words.append(sample["text"])
    tokens.append(tokenizer.tokenize(sample["text"]))

In [ ]:
print(len(tokens[0]))
print(len(words[0]))

In [ ]:
import matplotlib.pyplot as plt

# Compute the lengths of the texts (in characters) and their tokenizations (number of tokens)
text_lengths = [len(t) for t in words]
token_lengths = [len(tok) for tok in tokens]

# Compute average compression ratio: average number of characters per token
# (i.e., average compression ratio = average(original length / tokenized length))
compression_ratios = [tlen / tkl if tkl > 0 else 0 for tlen, tkl in zip(text_lengths, token_lengths)]
avg_compression_ratio = sum(compression_ratios) / len(compression_ratios)
print(f"Average compression ratio (characters per token): {avg_compression_ratio:.3f}")

plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
plt.hist(text_lengths, bins=200, color="skyblue", edgecolor="black", log=True)
plt.title("Distribution of Text Lengths (characters)")
plt.xlabel("Text length (characters)")
plt.ylabel("Count (log scale)")
plt.yscale('log')

plt.subplot(1,2,2)
plt.hist(token_lengths, bins=200, color="orange", edgecolor="black", log=True)
plt.title("Distribution of Tokenized Lengths (tokens)")
plt.xlabel("Tokenized length (tokens)")
plt.ylabel("Count (log scale)")
plt.yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
words = []
tokens = []

i = 0
for sample in hf_dataset["train"]:
    i += 1
    if i % 100 == 0:
        print(i)
    if i == 1000 :
        break
    words.extend([word for word in sample["text"].split()])

print(len(words))
words = set(words)
print(len(words))

for word in words:
    print(word)
    tokens.append(tokenizer.tokenize(word))



In [ ]:
import matplotlib.pyplot as plt

# Compute the number of tokens required for each word (fertility)
word_token_lengths = [len(toks) for toks in tokens]

# Compute the mean fertility
if word_token_lengths:
    mean_fertility = sum(word_token_lengths) / len(word_token_lengths)
else:
    mean_fertility = 0

print(f"Mean fertility (average # tokens per word): {mean_fertility:.3f}")

# Plot histogram and convert density to percentage
counts, bins, _ = plt.hist(
    word_token_lengths, 
    bins=range(1, max(word_token_lengths) + 2), 
    color='cadetblue', 
    edgecolor='black',
    density=True
)

# Convert density to percentage by multiplying by bin width and 100
bin_widths = [bins[i+1] - bins[i] for i in range(len(bins)-1)]
percentages = counts * bin_widths * 100

plt.clf()
plt.bar(bins[:-1], percentages, width=bin_widths, color='cadetblue', edgecolor='black', align='edge')
plt.title("Distribution of Tokenized Word Lengths (Fertility)")
plt.xlabel("Number of tokens per word")
plt.ylabel("Percentage (%)")
plt.xticks(range(1, max(word_token_lengths) + 1))
plt.show()

# Print percentage of "continuing words" (words that are split into more than one token)
n_words = len(word_token_lengths)
n_continuing = sum(1 for n in word_token_lengths if n > 1)
perc_continuing = (n_continuing / n_words) * 100 if n_words > 0 else 0
print(f"Percentage of continuing words (split into >1 token): {perc_continuing:.2f}%")


In [ ]:
ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", cache_dir=paths.DATA_DIR / "fineweb-edu").shuffle(0)


In [ ]:
# Compute fertility statistics for the first 1000 examples in the new dataset

all_words = []
all_tokens = []
fertilities = []

for i in range(10000):
    text = ds["train"][i]["text"]
    words = text.split()
    all_words.extend(words)
    # For each word, compute the fertility: number of tokens per word
    for word in words:
        tokens = tokenizer.tokenize(word)
        all_tokens.append(tokens)
        fertilities.append(len(tokens))

# Plot histogram of fertility per word
plt.figure(figsize=(8,5))
counts, bins, patches = plt.hist(
    fertilities, 
    bins=range(1, max(fertilities) + 2), 
    color='slateblue', 
    edgecolor='black',
    density=True
)
bin_widths = [bins[i+1] - bins[i] for i in range(len(bins)-1)]
percentages = counts * bin_widths * 100

plt.clf()
plt.bar(bins[:-1], percentages, width=bin_widths, color='slateblue', edgecolor='black', align='edge')
plt.title("Distribution of Fertility (Tokens per Word) (fineweb-edu, 1000 samples)")
plt.xlabel("Fertility (tokens per word)")
plt.ylabel("Percentage (%)")
plt.xticks(range(1, max(fertilities) + 1))
plt.show()

mean_fertility = sum(fertilities) / len(fertilities) if fertilities else 0
print(f"Mean fertility (average # tokens per word): {mean_fertility:.3f}")

# Percentage of words with fertility > 1
n_words = len(fertilities)
n_multi_token = sum(1 for n in fertilities if n > 1)
perc_multi_token = (n_multi_token / n_words) * 100 if n_words > 0 else 0
print(f"Percentage of words with fertility > 1: {perc_multi_token:.2f}%")


In [ ]:
import copy
import random
import re

ALPHA_WORD = re.compile(r"^[^\W\d_]+$", re.UNICODE)   # letters only, no digits/punct
STRIP_PUNCT = re.compile(r"^\W+|\W+$", re.UNICODE)


def prune_tokenizer(tokenizer, vocab_size):
    """Truncate the vocab to `vocab_size`, i.e. "what if training had stopped earlier"."""
    pruned = copy.copy(tokenizer)
    pruned.vocab_size = vocab_size
    pruned.vocab = dict(list(tokenizer.vocab.items())[:vocab_size])
    pruned.inv_vocab = {v: k for k, v in pruned.vocab.items()}
    return pruned


def fertility(tokenizer, texts):
    """Corpus-level cost + per-word morphological fertility.

    The Qwen pretokenizer absorbs the leading space into the word
    (`[^\r\n\p{L}\p{N}]?[\p{L}\p{M}]+`), so words are scored as " " + word --
    scoring the bare word measures a form the vocab rarely saw and inflates
    fertility by ~20%.

    Two different questions, two numbers:
      corpus_tokens_per_word -- true cost on raw text (punctuation and digits included)
      mean_fertility         -- vocab coverage of morphology (alphabetic words only,
                                so \p{N} digit-splitting doesn't contaminate it)
    """
    per_word = []
    total_tokens = 0
    total_words = 0
    for text in texts:
        total_tokens += len(tokenizer.tokenize(text))   # raw text, nothing removed
        for w in text.split():
            total_words += 1
            w = STRIP_PUNCT.sub("", w)
            if w and ALPHA_WORD.match(w):
                per_word.append(len(tokenizer.tokenize(" " + w)))
    n = len(per_word)
    return {
        "corpus_tokens_per_word": total_tokens / total_words if total_words else 0,
        "mean_fertility": sum(per_word) / n if n else 0,
        "pct_multi_token": 100 * sum(1 for f in per_word if f > 1) / n if n else 0,
        "fertilities": per_word,      # for the histogram below
    }


base_tokenizer = BytePairEncoding.from_file(
    paths.EXPERIMENTS_DIR / "classla_finewebedu_2gb_tokenizer_Bytes"
)
vocab_sizes = [32000, 48000, 64000, 100000]
N_SAMPLES = 1000

# Random-access N rows. Do NOT iterate ds["train"] -- it is 9.67M rows / ~27GB.
random.seed(0)
hr_split, en_split = hf_dataset["train"], ds["train"]
hr_sample_texts = hr_split.select(random.sample(range(len(hr_split)), N_SAMPLES))["text"]
en_sample_texts = en_split.select(random.sample(range(len(en_split)), N_SAMPLES))["text"]

fertility_stats = {"en": {}, "hr": {}}
for vocab_size in vocab_sizes:
    pruned = prune_tokenizer(base_tokenizer, vocab_size)
    for lang, texts in [("en", en_sample_texts), ("hr", hr_sample_texts)]:
        s = fertility(pruned, texts)
        fertility_stats[lang][vocab_size] = s
        print(
            f"[{lang.upper()}] vocab {vocab_size:>6,} | "
            f"corpus tok/word {s['corpus_tokens_per_word']:.3f} | "
            f"word fertility {s['mean_fertility']:.3f} | "
            f"% multi-token {s['pct_multi_token']:.2f}"
        )


In [ ]:

def prune_tokenizer(tokenizer, vocab_size):
    """Truncate the vocab to `vocab_size`, i.e. "what if training had stopped earlier"."""
    pruned = copy.copy(tokenizer)
    pruned.vocab_size = vocab_size
    pruned.vocab = dict(list(tokenizer.vocab.items())[:vocab_size])
    pruned.inv_vocab = {v: k for k, v in pruned.vocab.items()}
    return pruned

# Prune the tokenizer to 48k and save it
tokenizer_48k = prune_tokenizer(base_tokenizer, 48000)
tokenizer_48k.save(paths.EXPERIMENTS_DIR / "classla_finewebedu_2gb_tokenizer_Bytes_48k")

In [ ]:
tokenizer_48k = BytePairEncoding.from_file(paths.EXPERIMENTS_DIR / "gpt2-test-large/BPE_vocab_48k")

In [ ]:
text = " velik, veliki, mali, brojevi, broj, broja, brod, broju, brojem, brojom, brojevima, brojeva, brojevima, velikoga, velikome, velikima, maloga, malima"
[(tokenizer.decode(token), token) for token in tokenizer_48k.tokenize(text)]
